# DSML 4220 - Lab 10: A simple Agent with Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

In this lab we will use Ollama to create a simple agent armed with tools in order to help carry out tasks on our behalf. This notebook is based on the short blog posts/tutorials found [here](https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide) and [here](https://towardsdatascience.com/ai-agents-from-zero-to-hero-part-1/).


### Lab 10 Assignment/Task
There are a few questions below that require some additional code to be written so that your agent can carry out other operations besides just addition.

Let's start out by setting up Ollama to run in Colab. If you run this notebook locally and already have Ollama running, then you can skip these steps.

In [1]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,594 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

The following two modules we'll need later on, but we install them here because Colab may ask to restart after they are installed with `pip`. It's better to restart at the beginning than to restart half-way through.

In [2]:
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 7.3 MB/s eta 0:00:00


Now we need to get the Ollama server running. Run the following code block to do this.

In [3]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Next, let's pull the model we want to use, Llama 3.2 with 1 billion parameters.

In [4]:
!ollama pull llama3.2:1b

Then, install the Ollama Python api.

In [5]:
!pip install ollama

Finally, get started with using Ollama from Python.

In [6]:
import ollama

Now, let's define a __tool__ for the agent/model to use.

In [13]:
# Tool function to add two numbers
def add_two_numbers(a: int, b: int) -> int:
    return int(a) + int(b)

Next, let's set up the system prompt and an initial user prompt/question for the agent/model.

In [14]:
# System prompt to inform the model about the tool is usage
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."
}

# A sample of user input asking a math question
user_message = {
    "role": "user",
    "content": "What is 90999999 + 10000001?"
}

messages = [system_message, user_message]
messages

[{'role': 'system',
  'content': "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."},
 {'role': 'user', 'content': 'What is 90999999 + 10000001?'}]

Ask the agent/model to respond.

In [15]:
# Ask llama3.2 to respond
response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers]
)

In [16]:
response.message

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='add_two_numbers', arguments={'a': '90999999', 'b': '10000001'}))])

In [17]:
response.message.content

''

In [18]:
# Check if the model called a function
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        func_name = tool_call.function.name   # e.g., "add_two_numbers"
        args = tool_call.function.arguments   # e.g., {"a": 10, "b": 10}
        # If the function name matches and we have it in our tools, execute it:
        if func_name == "add_two_numbers":
            result = add_two_numbers(**args)
            print("Function output:", result)




Function output: 101000000


---

### Q1: Does the above output look correct? Does it look like the sum of the numbers 90999999 and 10000001? Why is it not correct?

(Hint: there is nothing wrong with the model/agent here, but rather the tool implementation; namely, Python's [type hints](https://docs.python.org/3/library/typing.html) are not a guarantee that the correct/intended data type is used, so you may need to add some type casting inside of the function `add_two_numbers`)

`<The function the model used simply concatenated the two numbers instead of algebraically adding them. This is where the type cast change comes in, and I will impliment it now. >`

---

In [19]:
# Complete the agent's tool call and allow the model to use output to formulate an answer
""" (Continuing from previous code) """
available_functions = {"add_two_numbers": add_two_numbers}#, "multiply_two_numbers": multiply_two_numbers}

""" System prompt to inform the model about the tool is usage """

""" Model's initial response after possibly invoking the tool """
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

""" If a tool was called, handle it """
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): I can summarise the calculation as follows:

90999999 + 10000001 = 101000000.

The final answer is 101000000.


---

### Q2: Try running the code cell below. Does it return the expect result? If note, then add/modify the necessary code to allow Llama3.2 to use its  multiplication tool. Then rerun your code cell below; now did it output the expected result?

`<INSERT YOUR ANSWER HERE>`

---

In [23]:
# Implement a multiplication function by replacing the `pass` statement below with the correct return statement
def multiply_two_numbers(a: int, b: int) -> int:
    return int(a) * int(b)


""" System prompt to inform the model about the tool is usage """
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do addition by calling the function 'add_two_numbers' or multiplication by calling the function 'multiply_two_numbers'."
}
# User asks a question that involves a calculation
user_message = {
    "role": "user",
    "content": "What is 10001 multiplied by 6?"
}

messages = [system_message, user_message]

response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers, multiply_two_numbers]  # pass the actual function object as a tool
)

# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): To find the result of multiplying 10001 by 6, I added two numbers together.

1. First, I called the function 'multiply_two_numbers' with the arguments 10001 and 6.
2. The result of this function is 60006.
3. Since the original task was to multiply 10001 by 6, which can also be done by calling the function 'multiply_two_numbers', my previous response was incorrect.

The correct results are:

- Multiplying 10001 by 6 using a function: 60006
- Using two separate functions (addition and multiplication): Not necessary in this case.


In [24]:
follow_up.message

Message(role='assistant', content="To find the result of multiplying 10001 by 6, I added two numbers together.\n\n1. First, I called the function 'multiply_two_numbers' with the arguments 10001 and 6.\n2. The result of this function is 60006.\n3. Since the original task was to multiply 10001 by 6, which can also be done by calling the function 'multiply_two_numbers', my previous response was incorrect.\n\nThe correct results are:\n\n- Multiplying 10001 by 6 using a function: 60006\n- Using two separate functions (addition and multiplication): Not necessary in this case.", thinking=None, images=None, tool_name=None, tool_calls=None)

Next let's equip our agent to retrieve external information, which will require a few more tools to be able to search the web.

In [59]:
from langchain_community.tools import DuckDuckGoSearchResults

def search_web(query: str) -> str:
  return DuckDuckGoSearchResults(backend="news").run(query)

tool_search_web = {
    'type': 'function',
    'function': {
        'name': 'search_web',
        'description': 'Search the web',
        'parameters': {
            'properties': {
                'query': {'type':'string', 'description':'the topic or subject to search on the web'}
            },
            'required': ['query']
        }
    }
}

# Quickly test and see what a general web news search for Los Angeles yields
search_web(query="Los Angeles")

"snippet: Los Angeles Mayor Karen Bass has announced plans to streamline building permits to speed up housing..., title: Los Angeles Mayor Karen Bass To Speed Up Permits After Meeting With Trump, link: https://www.yahoo.com/news/articles/los-angeles-mayor-karen-bass-183427740.html, date: 2026-04-28T18:34:04+00:00, source: Realtor.com · via Yahoo News, snippet: MLB.com polled 17 anonymous MLB executives on the winner of each division, and all 17 voted the Los..., title: Dodgers Unanimously Predicted to Win Division by MLB Execs, link: https://sports.yahoo.com/articles/dodgers-unanimously-predicted-win-division-185345437.html, date: 2026-04-28T18:54:04+00:00, source: Dodgers Nation · via Yahoo Sports, snippet: Court documents, obtained The California Post, show the NFL star sued the three entities in Los Ange..., title: NFL star Jameson Williams files bombshell lawsuit against NCAA, Big Ten and SEC, link: https://nypost.com/2026/04/28/sports/lions-receiver-jameson-williams-sues-ncaa-big-

In [60]:
def search_ys(query: str) -> str:
  engine = DuckDuckGoSearchResults(backend="news")
  return engine.run(f"site:sports.yahoo.com {query}")

tool_search_ys = {
    'type': 'function',
    'function': {
        'name': 'search_ys',
        'description': 'Search for sports news',
        'parameters': {
            'properties': {
                'query': {'type':'string', 'description':'the sport, sports team, or subject to search'}
            },
            'required': ['query']
        }
    }
}

# Quickly test and see what a search for Los Angeles in the sports section of the news yields
search_ys(query="Los Angeles")

"snippet: The Los Angeles Rams eschewed adding a difference-maker for a Super Bowl run in 2026 in the 2026 NFL Draft, so their draft ..., title: Los Angeles Rams Day 3 2026 NFL Mock Draft: Rams ran out of time to add a 2026 difference-maker, so it's all about depth, link: https://sports.yahoo.com/articles/los-angeles-rams-day-3-110000388.html, date: 2026-04-25T19:02:05+00:00, source: Yahoo Sports, snippet: The Los Angeles Chargers addressed several needs during the 2026 NFL draft. Here's an early look at where they fit on the ..., title: Los Angeles Chargers' 2026 depth chart, signings and best available free agents, link: https://sports.yahoo.com/articles/los-angeles-chargers-2026-depth-020407764.html, date: 2026-04-26T19:02:05+00:00, source: Yahoo Sports, snippet: The Los Angeles Rams have routinely struck gold in the undrafted free agent process over the years. Who did they add ..., title: Los Angeles Rams UDFA Tracker: Rams complement small draft with massive haul of undrafted tale

In [61]:
system_message = {
    "role": "system",
    "content": "You are a helpful assistant with access to tools for search the web for current news and events."
    }
user_message = {
    "role": "user",
    "content": "Tell me about sports teams in the city of Denver" # YOU WILL CHANGE THIS QUESTION, SEE Q3 BELOW
}
messages = [system_message, user_message]

In [62]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant with access to tools for search the web for current news and events.'},
 {'role': 'user',
  'content': 'Tell me about sports teams in the city of Denver'}]

In [63]:
response = ollama.chat(
  model="llama3.2:1b",
  tools=[tool_search_web, tool_search_ys],
  messages=messages
)
response

ChatResponse(model='llama3.2:1b', created_at='2026-04-28T19:02:10.527290743Z', done=True, done_reason='stop', total_duration=442849153, load_duration=193687844, prompt_eval_count=225, prompt_eval_duration=16992333, eval_count=21, eval_duration=178653902, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_ys', arguments={'query': 'sports teams in Denver'}))]), logprobs=None)

In [64]:
# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {'search_web':search_web, 'search_ys':search_ys}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        # Preprocess arguments: remove '@' if it exists in the key
        processed_args = {}
        for key, value in tool_call.function.arguments.items():
            if key.startswith('@'):
                processed_args[key[1:]] = value
            else:
                processed_args[key] = value

        result = func(**processed_args)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): After conducting a web search for news and events related to sports teams in Denver, I was unable to find any information on specific sports teams in Denver. However, I did find that the Denver Nuggets are a professional basketball team that plays in the National Basketball Association (NBA), and they are based in Denver, Colorado.

If you're looking for information on sports teams in the surrounding area, I found that there are several other teams that play in nearby cities, including:

* The Colorado Rockies (Major League Baseball) play in Denver's Coors Field.
* The Colorado Avalanche (National Hockey League) also play in Denver at the Pepsi Center.
* The Colorado Rapids (Major League Soccer) play their home games at Dick's Sporting Goods Park.

I was unable to find any information on sports teams that are based directly in Denver.


---

### Q3: The question above currently asks about Denver, but change the question to include a word or reference to sports. Does the agent use the correct tool based on your prompt/question? Be sure to also run the code cells above with your modified promp/question.

`<After some prompt tweaking I was able to get it to return information about sports teams in denver by conducting an internet search!>`

---